# WorldQuant Brain — Alpha Analysis

**This notebook is read-only with respect to simulations.**  
Run `python main.py` first to simulate and save results, then use this notebook to explore them.

---

## 0. Setup

In [ ]:
import sys
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Add parent directory so we can import project modules
sys.path.insert(0, os.path.abspath('..'))

from results import compare_alphas, fetch_all_recordsets
from cache import cache_stats
from auth import create_session

# Plotting style
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 13

RESULTS_DIR = '../results/'
print('Setup complete.')

## 1. Load Last Run

In [ ]:
manifest_path = os.path.join(RESULTS_DIR, 'last_run.json')

with open(manifest_path) as f:
    manifest = json.load(f)

all_alpha_ids = manifest['simulated'] + manifest['cached']

print(f"Last run summary:")
print(f"  Newly simulated : {len(manifest['simulated'])}")
print(f"  From cache      : {len(manifest['cached'])}")
print(f"  Failed          : {manifest['failed_count']}")
print(f"  Total to analyse: {len(all_alpha_ids)}")
print()
print('Alpha IDs:', all_alpha_ids)

## 2. Load Saved Results

In [ ]:
def load_recordset(alpha_id: str, recordset: str) -> pd.DataFrame:
    """Load a previously saved recordset from disk."""
    path = os.path.join(RESULTS_DIR, alpha_id, f'{recordset}.json')
    if not os.path.exists(path):
        print(f'  Missing: {path}')
        return pd.DataFrame()
    df = pd.read_json(path)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    return df

# Load PnL and Sharpe for all alphas
pnl_data    = {aid: load_recordset(aid, 'pnl')          for aid in all_alpha_ids}
sharpe_data = {aid: load_recordset(aid, 'sharpe')       for aid in all_alpha_ids}
yearly_data = {aid: load_recordset(aid, 'yearly-stats') for aid in all_alpha_ids}

print('Data loaded for', len(pnl_data), 'alphas.')

## 3. PnL Curves — All Alphas

In [ ]:
fig, ax = plt.subplots()

for alpha_id, df in pnl_data.items():
    if df.empty:
        continue
    pnl_col = [c for c in df.columns if c != 'date'][0]
    ax.plot(df['date'], df[pnl_col], label=alpha_id[:12] + '...')

ax.set_title('Cumulative PnL — All Alphas')
ax.set_xlabel('Date')
ax.set_ylabel('PnL')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

## 4. Sharpe Ratio Comparison

In [ ]:
# Build a summary table: final Sharpe for each alpha
rows = []
for alpha_id, df in sharpe_data.items():
    if df.empty:
        continue
    sharpe_col = [c for c in df.columns if c != 'date'][0]
    final_sharpe = df[sharpe_col].iloc[-1]
    rows.append({'alpha_id': alpha_id, 'final_sharpe': round(final_sharpe, 4)})

summary = pd.DataFrame(rows).sort_values('final_sharpe', ascending=False)
display(summary)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['steelblue' if s >= 0 else 'firebrick' for s in summary['final_sharpe']]
ax.barh([aid[:16] for aid in summary['alpha_id']], summary['final_sharpe'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Final Sharpe Ratio by Alpha')
ax.set_xlabel('Sharpe')
plt.tight_layout()
plt.show()

## 5. Yearly Stats Breakdown

In [ ]:
# Show yearly stats for a specific alpha — change the index as needed
ALPHA_TO_INSPECT = all_alpha_ids[0] if all_alpha_ids else None

if ALPHA_TO_INSPECT:
    df = yearly_data[ALPHA_TO_INSPECT]
    print(f'Yearly stats for: {ALPHA_TO_INSPECT}')
    display(df)
else:
    print('No alphas to inspect.')

## 6. Cache Status

In [ ]:
stats = cache_stats()
print(f"Total cached simulations : {stats['total']}")
print(f"Oldest entry             : {stats['oldest']}")
print(f"Newest entry             : {stats['newest']}")

## 7. Fetch Fresh Data (Optional — requires live session)

Use this section only when you need data not saved by `main.py`.  
Avoid running simulations here.

In [ ]:
# Uncomment to authenticate and fetch live recordsets
# session = create_session()

# alpha_id = 'paste-an-alpha-id-here'
# fresh = fetch_all_recordsets(session, alpha_id, ['pnl', 'sharpe', 'turnover'])
# display(fresh['pnl'].head())